In [5]:
# build_long_micro_panel.py
"""
(1) Skeleton : 영화 × 전날짜(2020-01-22 ~ 2025-06-16)
(2) Reviews  : 리뷰 원자료(작성일 기준) 그대로 유지
(3) LEFT MERGE → Skeleton 행이 리뷰 수만큼 복제된 long-micro 패널

출력: movie_long_micro_panel.parquet
"""

import pandas as pd
from pathlib import Path

# ──────────────────────────────────────────────────────────────────────
# 0. 설정
XL_PATH   = "merged_reviews_updated_v3(fixed_dates).xlsx"
OUT_PATH  = "movie_long_micro_panel.parquet"
DATE_START = pd.Timestamp("2020-01-22")
DATE_END   = pd.Timestamp("2025-06-16")

# ──────────────────────────────────────────────────────────────────────
# 1. 공통: 엑셀 serial → datetime 안전 변환
def excel_serial_to_datetime(v):
    if pd.isna(v):
        return pd.NaT
    if isinstance(v, (int, float)):
        return pd.Timestamp("1899-12-30") + pd.to_timedelta(int(v), "D")
    return pd.to_datetime(v, errors="coerce")

# ──────────────────────────────────────────────────────────────────────
# 2. Skeleton (영화별 1행 → 전날짜 생성)
meta_cols = ["영화명", "개봉일", "넷플릭스 공개일", "대표국적", "장르"]
meta = (
    pd.read_excel(XL_PATH, usecols=meta_cols, engine="openpyxl")
      .drop_duplicates(subset=["영화명"])                      # 영화별 1행
)

meta["개봉일"]        = meta["개봉일"].apply(excel_serial_to_datetime)
meta["넷플릭스 공개일"] = meta["넷플릭스 공개일"].apply(excel_serial_to_datetime)

# 모든 영화에 동일한 날짜 range 부여 후 explode
full_range = pd.date_range(DATE_START, DATE_END, freq="D")
meta["일자"] = [full_range] * len(meta)

skeleton = (
    meta
    .explode("일자", ignore_index=True)
    .assign(
        post_netflix=lambda d: (
            d["일자"] >= d["넷플릭스 공개일"]
        ).astype("int8")
    )
    .drop(columns="넷플릭스 공개일")               # 필요 시 보존 가능
    .astype({"대표국적": "category", "장르": "category"})
)

# ──────────────────────────────────────────────────────────────────────
# 3. 리뷰 원자료 로드 (작성일 = 일자)
rev_cols = [
    "영화명", "작성일",          # key
    "작성자", "감상평", "별점", "공감수", "비공감수",
    "실관람객", "OTT관람객"      # ← 필요 열 자유롭게 추가/제거
]
reviews = pd.read_excel(XL_PATH, usecols=rev_cols, engine="openpyxl")
reviews["작성일"] = reviews["작성일"].apply(excel_serial_to_datetime)
reviews = reviews.rename(columns={"작성일": "일자"})

# ──────────────────────────────────────────────────────────────────────
# 4. LEFT MERGE  (Skeleton 행이 리뷰 수만큼 복제)
panel_long = (
    skeleton
    .merge(
        reviews,
        on=["영화명", "일자"],
        how="left",              # 리뷰 없는 날짜 → NaN 한 행
        validate="1:m"           # 1(스켈톤) : many(리뷰) 관계
    )
    .sort_values(["영화명", "일자"])
    .reset_index(drop=True)
)

# ──────────────────────────────────────────────────────────────────────
# 5. 메모리 최적화 & 저장
panel_long["영화명"] = panel_long["영화명"].astype("category")
panel_long.to_parquet(OUT_PATH, index=False)
print(f"saved  ➜  {OUT_PATH}\nshape ➜  {panel_long.shape}")


saved  ➜  movie_long_micro_panel.parquet
shape ➜  (437133, 13)


In [13]:
import pandas as pd

parquet_path = "movie_long_micro_panel.parquet"
csv_path     = "movie_long_micro_panel.csv"       # .csv.gz 도 가능

# Parquet → DataFrame
df = pd.read_parquet(parquet_path)

# CSV 저장 (UTF-8-SIG: 엑셀 한글 깨짐 방지)
df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("CSV 저장 완료:", csv_path)

CSV 저장 완료: movie_long_micro_panel.csv
